In [ ]:
import xarray as xr
import numpy as np
import functions
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import matplotlib as mpl
import seaborn as sns
import cmcrameri
from scipy import stats
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.markers import MarkerStyle
import statsmodels.api as sm
import pymannkendall as mk
from global_land_mask import globe

import xesmf as xe

In [19]:
rpath = '/nird/datapeak/NS9600K/astridbg/arctic-cld-feedbacks/data/observations_reanalysis_data/'

## Processing and re-indexing of data

### Snow cover

In [ ]:
filepath = rpath+'snow_cover/nhsce_v01r01_19661004_20260202.nc'
sce = xr.open_dataset(filepath)

# Select times with meaningful data
sce = sce.sel(time=slice('1975-01-01','2026-01-01'))

# Rename longitude and latitude and set to coordinates
sce = sce.rename({'latitude': 'lat', 'longitude':'lon'})
sce = sce.set_coords(['lat', 'lon'])

# Create a target latitude-longitude grid
resolution = 0.5
target_lat = np.arange(0, 90 + resolution, resolution)
target_lon = np.arange(-180, 180, resolution)

target_grid = xr.Dataset(
    coords={
        "lat": ("lat", target_lat),
        "lon": ("lon", target_lon),
    }
)

# Regrid data using nearest neighbor
regridder = xe.Regridder(
    sce,
    target_grid,
    method="nearest_s2d",
)

ds_nearest = regridder(sce)

# Remove low latitudes regions
ds_nearest = ds_nearest.sel(lat=slice(40,90))
ds_nearest.to_netcdf(rpath+'snow_cover/nhsce_regridded_latlon.nc')


### Land surface temperature

In [15]:
filepath = rpath + 'temperature/ERA5-Land_t2m_Arctic.nc'
temp_ERA5_Land = xr.open_dataset(filepath)

# Re-index
temp_ERA5_Land = temp_ERA5_Land.rename({'latitude':'lat', 'longitude':'lon', 'valid_time':'time'})
temp_ERA5_Land = temp_ERA5_Land.reindex(lat=list(reversed(temp_ERA5_Land.lat)))
lons = np.array(temp_ERA5_Land.coords['lon'])
lons[np.where(lons<0)] = 360 + lons[np.where(lons<0)]
temp_ERA5_Land.coords['lon'] = lons
temp_ERA5_Land = temp_ERA5_Land.sortby(temp_ERA5_Land.lon)

temp_ERA5_Land.to_netcdf(rpath+'temperature/ERA5-Land_t2m_Arctic_reindexed.nc')

### Soil moisture ERA5-Land

In [ ]:
# OPEN DATASET

filepath = rpath + 'soil_moisture/ERA5-Land_swvl1_swvl2.nc'
SM_ERA5 = xr.open_dataset(filepath)

<xarray.Dataset> Size: 1GB
Dimensions:     (valid_time: 171, latitude: 301, longitude: 3600)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 1kB 1970-06-01 ... 2026-08-01
    expver      (valid_time) <U4 3kB ...
  * latitude    (latitude) float64 2kB 90.0 89.9 89.8 89.7 ... 60.2 60.1 60.0
  * longitude   (longitude) float64 29kB -180.0 -179.9 -179.8 ... 179.8 179.9
    number      int64 8B ...
Data variables:
    swvl1       (valid_time, latitude, longitude) float32 741MB ...
    swvl2       (valid_time, latitude, longitude) float32 741MB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-09-21T16:05 GRIB to CDM+CF via cfgrib-0.9.1...

In [24]:
# CREATE SOIL LEVEL FOR 10 CM

# Soil level 1: 0-7 cm
# Soil level 2: 7-28 cm
weighting_fraction = (10-7)/(28-7)
SM_ERA5['swvl_10cm'] = SM_ERA5['swvl1']+weighting_fraction*SM_ERA5['swvl2']

In [25]:
# CREATE SOIL LEVEL FOR 20 CM

# Soil level 1: 0-7 cm
# Soil level 2: 7-28 cm
weighting_fraction = (20-7)/(28-7)
SM_ERA5['swvl_20cm'] = SM_ERA5['swvl1']+((20-7)/(28-7))*SM_ERA5['swvl2']

In [ ]:
# RE-INDEX

SM_ERA5 = SM_ERA5.rename({'latitude':'lat', 'longitude':'lon', 'valid_time':'time'})
SM_ERA5_1970_2025 = SM_ERA5.sel(time=slice('1970-01-01','2025-09-01'))
SM_ERA5_1970_2025 = SM_ERA5_1970_2025.reindex(lat=list(reversed(SM_ERA5_1970_2025.lat)))
lons = np.array(SM_ERA5_1970_2025.coords['lon'])
lons[np.where(lons<0)] = 360 + lons[np.where(lons<0)]
SM_ERA5_1970_2025.coords['lon'] = lons
SM_ERA5_1970_2025 = SM_ERA5_1970_2025.sortby(SM_ERA5_1970_2025.lon)
SM_ERA5_1970_2025.to_netcdf(rpath+'soil_moisture/SM_ERA5_1970_2025_10cm_20cm.nc')

### Soil moisture Wang & Mao (2021)

In [57]:
filepath='/nird/datalake/NS9560K/diagnostics/ILAMB-Data/DATA/mrsos/WangMao/mrsos_olc.nc'

SM_WM = xr.open_dataset(filepath)
SM_WM_1970_2016 = SM_WM.sel(time=slice('1970-01-01','2016-12-31'),lat=slice(59,90))
SM_WM_1970_2016_masked = SM_WM_1970_2016.where(SM_WM_1970_2016['mrsos'] < 1000)
SM_WM_1970_2016_masked = SM_WM_1970_2016_masked.drop_vars('time_bounds')
lons = np.array(SM_WM_1970_2016_masked.coords['lon'])
lons[np.where(lons<0)] = 360 + lons[np.where(lons<0)]
SM_WM_1970_2016_masked.coords['lon'] = lons
SM_WM_1970_2016_masked = SM_WM_1970_2016_masked.sortby(SM_WM_1970_2016_masked.lon)
SM_WM_1970_2016_masked.to_netcdf(rpath+'SM_WangMao_1970_2016_masked.nc')